In [ ]:
import os 
import torch 
import torch.nn as nn
import torchvision.models  as models 
import torchvision.transforms as transforms 
from PIL import Image 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"-->Execution target confirmed: {device}")
if torch.cuda.is_available(): 
    print(f"--> Cloud Hardware Profile: {torch.cuda.get_device_name(0)}") 
    
backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

modules = list(backbone.children())[:-2]
feature_extractor = nn.Sequential(*modules).to(device) 
feature_extractor.eval() 

print("--> Pre-trained ResNet-50 backbone loaded. Classification head pruned sucessfully")

In [5]:
import torch.nn as nn 
import torch.nn.functional as F 

class SaliencyDecoder(nn.Module): 
    def __init__(self): 
        super(SaliencyDecoder, self).__init__() 
        self.conv1 = nn.Conv2d(2048, 512, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(512, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 1, kernel_size=1)
        
    def forward(self, x): 
        x = F.relu(self.conv1(x)) 
        x = F.interpolate(x, scale_factor=4, mode='bilinear', align_corners=False)
        x = F.relu(self.conv2(x))
        x = F.interpolate(x, scale_factor=8, mode='bilinear', align_corners=False) 
        saliency_map = torch.sigmoid(self.conv3(x))
        return saliency_map

decoder = SaliencyDecoder().to(device)
print("--> Saliency Decoder initialized and pushed to Tesla T4 GPU.")

--> Saliency Decoder initialized and pushed to Tesla T4 GPU.
